# 🔍 Depth Estimation Pipeline — MiDaS Edition
**YOLOv8n + MiDaS (DPT) + DINOv2**

| Model | HuggingFace ID | Size | Speed |
|-------|---------------|------|-------|
| MiDaS DPT-Hybrid | `Intel/dpt-hybrid-midas` | ~400 MB | Fast |
| MiDaS DPT-Large  | `Intel/dpt-large`        | ~800 MB | Slower, better quality |

Works on: ✅ Colab T4 GPU · ✅ Colab CPU · ✅ Mac M1/M2/M3 (MPS)

> **Colab tip:** Runtime → Change runtime type → **T4 GPU** for best speed

In [1]:
# ── Cell 1: Detect environment ────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Environment : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Device      : {DEVICE}')
if DEVICE == 'cpu':
    print('  ⚠️  CPU mode — expect ~5-10 min per image. Consider enabling T4 GPU.')
elif DEVICE == 'mps':
    print('  ✅ Apple Silicon MPS detected')

Environment : Google Colab
Device      : cuda


In [2]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('torch>=2.1.0', 'torchvision>=0.16.0')
pip('transformers>=4.40.0', 'huggingface_hub>=0.22.0')
pip('ultralytics>=8.2.0')
pip('opencv-python-headless>=4.9.0')
pip('Pillow>=10.0.0', 'numpy>=1.24.0', 'matplotlib>=3.8.0', 'tqdm')

print('\n✅ All packages installed')


✅ All packages installed


In [3]:
# ── Cell 3: Upload depth_pipeline.py ─────────────────────────────────────────
# Upload the NEW depth_pipeline.py (MiDaS version) from your Downloads.
# Colab sometimes renames duplicates to 'depth_pipeline (2).py' — handled below.
# Local users: skip this cell if depth_pipeline.py is already here.

import sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('Upload depth_pipeline.py (the .py file, not the notebook) …')
    uploaded = files.upload()

    # Match any variant Colab might rename it to
    py_files = [f for f in uploaded if 'depth_pipeline' in f and f.endswith('.py')]
    if not py_files:
        raise FileNotFoundError(
            'No depth_pipeline*.py found — please upload depth_pipeline.py'
        )
    uploaded_name = py_files[0]
    if uploaded_name != 'depth_pipeline.py':
        shutil.copy(uploaded_name, 'depth_pipeline.py')
        print(f'  (renamed {uploaded_name!r} → depth_pipeline.py)')
    print('✅ depth_pipeline.py ready')
else:
    if not Path('depth_pipeline.py').exists():
        raise FileNotFoundError('depth_pipeline.py not found in current folder.')
    print('✅ depth_pipeline.py found locally')

Upload depth_pipeline.py (the .py file, not the notebook) …


Saving depth_pipeline.py to depth_pipeline.py
✅ depth_pipeline.py ready


In [4]:
# ── Cell 4: Upload dataset images ────────────────────────────────────────────
# Creates dataset/ and moves your images into it.
# Local users: make sure your images are in a folder called dataset/ and skip.

import os, shutil, sys
from pathlib import Path

DATASET_DIR = Path('dataset')
DATASET_DIR.mkdir(exist_ok=True)

IN_COLAB = 'google.colab' in sys.modules
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

if IN_COLAB:
    from google.colab import files
    print('Select your images (jpg/png/webp etc) — you can pick multiple:')
    uploaded = files.upload()

    moved = []
    for fname in uploaded:
        if Path(fname).suffix.lower() in IMG_EXTS:
            dest = DATASET_DIR / fname
            shutil.move(fname, dest)
            moved.append(dest.name)

    print(f'\n✅ {len(moved)} image(s) ready in dataset/:')
    for m in moved:
        print(f'   {m}')
else:
    imgs = [f for f in DATASET_DIR.glob('*') if f.suffix.lower() in IMG_EXTS]
    if not imgs:
        print('⚠️  dataset/ is empty — add images and re-run')
    else:
        print(f'✅ {len(imgs)} image(s) found in dataset/:')
        for f in imgs:
            print(f'   {f.name}')

Select your images (jpg/png/webp etc) — you can pick multiple:


Saving IMG_0557.JPG to IMG_0557.JPG
Saving IMG_0558.jpg to IMG_0558.jpg
Saving IMG_0559.JPG to IMG_0559.JPG
Saving IMG_0560.jpg to IMG_0560.jpg
Saving IMG_6324.heic to IMG_6324.heic
Saving IMG_6325.heic to IMG_6325.heic
Saving IMG_6326.heic to IMG_6326.heic
Saving IMG_6327.heic to IMG_6327.heic

✅ 4 image(s) ready in dataset/:
   IMG_0557.JPG
   IMG_0558.jpg
   IMG_0559.JPG
   IMG_0560.jpg


In [5]:
# ── Cell 5: Run the pipeline ──────────────────────────────────────────────────
# MiDaS models are downloaded from HuggingFace on first run and cached.
#
# To switch to the larger/better MiDaS model, change --depth to:
#   Intel/dpt-large   (needs GPU, ~800 MB)

import sys
sys.argv = [
    'depth_pipeline.py',
    '--input',    'dataset',
    '--output',   'output',
    '--depth',    'Intel/dpt-hybrid-midas',   # or Intel/dpt-large
    '--dino',     'facebook/dinov2-base',
    '--colormap', 'inferno',
    # '--device', 'mps',   # ← uncomment when running on Mac M1/M2/M3 locally
]

exec(open('depth_pipeline.py').read())


════════════════════════════════════════════════════════════
  Depth Estimation Pipeline  (MiDaS edition)
  Device  : cuda
  YOLO    : yolov8n.pt
  Depth   : Intel/dpt-hybrid-midas
  DINOv2  : facebook/dinov2-base
════════════════════════════════════════════════════════════

[Loading models]
  ▶ YOLOv8n   loading 'yolov8n.pt' …
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  ▶ MiDaS     loading 'Intel/dpt-hybrid-midas' …


preprocessor_config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/490M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/414 [00:00<?, ?it/s]

...

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


[Models ready]

Found 4 image(s)  →  output: 'output'

────────────────────────────────────────────────────────────
  IMG_0557.JPG
  [1/3] MiDaS depth … done  (min=0.00, max=3686.62)
  [2/3] YOLOv8n detection … done  (11 object(s) detected)
  [3/3] DINOv2 features … done
  Detections:
  [car            ]  conf=0.74  bbox=(1445,2734,1982,2939)  depth_median=0.1524  depth_std=0.0059  dino‖feat‖=48.20
  [car            ]  conf=0.70  bbox=(3011,2732,3245,2925)  depth_median=0.1363  depth_std=0.0059  dino‖feat‖=48.29
  [car            ]  conf=0.69  bbox=(2075,2724,2293,2913)  depth_median=0.1423  depth_std=0.0042  dino‖feat‖=47.27
  [car            ]  conf=0.63  bbox=(2398,2744,2661,2927)  depth_median=0.1488  depth_std=0.0049  dino‖feat‖=47.57
  [car            ]  conf=0.60  bbox=(3640,2742,3975,2927)  depth_median=0.1434  depth_std=0.0056  dino‖feat‖=47.18
  [car            ]  conf=0.53  bbox=(846,2739,1139,2928)  depth_median=0.1464  depth_std=0.0056  dino‖feat‖=44.14
  [car            

In [6]:
# ── Cell 6: Preview results inline ───────────────────────────────────────────
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

composites = sorted(Path('output').glob('*_composite.jpg'))

if not composites:
    print('No composites found — did Cell 5 complete successfully?')
else:
    for comp in composites:
        img = mpimg.imread(str(comp))
        fig, ax = plt.subplots(figsize=(18, 5))
        ax.imshow(img)
        ax.set_title(comp.stem, fontsize=12, pad=8)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print()

In [7]:
# ── Cell 7: Show JSON results ─────────────────────────────────────────────────
import json
from pathlib import Path

json_path = Path('output/results.json')
if json_path.exists():
    data = json.loads(json_path.read_text())
    for item in data:
        print(f"\n📷 {item['file']}  ({item['image_size'][0]}x{item['image_size'][1]})")
        print(f"   Detections: {item['n_detections']}")
        for d in item['detections']:
            print(f"   • {d['class']:<15s}  conf={d['confidence']:.2f}  "
                  f"depth_median={d['depth_median']:.4f}  "
                  f"dino_norm={d['dino_feat_norm']:.2f}")
else:
    print('results.json not found — run Cell 5 first.')


📷 IMG_0557.JPG  (4110x5480)
   Detections: 11
   • car              conf=0.74  depth_median=0.1524  dino_norm=48.20
   • car              conf=0.70  depth_median=0.1363  dino_norm=48.29
   • car              conf=0.69  depth_median=0.1423  dino_norm=47.27
   • car              conf=0.63  depth_median=0.1488  dino_norm=47.57
   • car              conf=0.60  depth_median=0.1434  dino_norm=47.18
   • car              conf=0.53  depth_median=0.1464  dino_norm=44.14
   • car              conf=0.52  depth_median=0.1446  dino_norm=45.19
   • car              conf=0.49  depth_median=0.1500  dino_norm=42.40
   • car              conf=0.42  depth_median=0.1363  dino_norm=44.81
   • car              conf=0.34  depth_median=0.1419  dino_norm=43.57
   • car              conf=0.32  depth_median=0.1446  dino_norm=41.45

📷 IMG_0558.jpg  (4052x5403)
   Detections: 10
   • car              conf=0.72  depth_median=0.1813  dino_norm=47.95
   • car              conf=0.69  depth_median=0.1791  dino_norm=48

In [8]:
# ── Cell 8: Download all outputs (Colab only) ─────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import shutil
    shutil.make_archive('depth_pipeline_output', 'zip', 'output')
    from google.colab import files
    files.download('depth_pipeline_output.zip')
    print('✅ Download started')
else:
    print('Local run — results are in the output/ folder.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started
